# 11 — Junk Dimension — Spark SQL

Combinações de flags geradas via Python + carregadas com `INSERT INTO`.
`ALTER TABLE ADD COLUMNS` e `MERGE INTO` em SQL para popular a FK no FactSales.

In [1]:
import sys
import os
sys.path.insert(0, os.getcwd())
from utils import get_spark, register_catalog, WAREHOUSE_DIR
import itertools
from pyspark.sql.types import StructType, StructField, StringType, BooleanType, IntegerType

spark = get_spark("NorthwindDW SQL - 11 Junk Dimension")
print("Spark:", spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/03/29 00:58:57 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark: 3.5.0


In [2]:
register_catalog(spark)

Catálogo registrado: {'bronze': 11, 'silver': 0, 'gold': 12}


In [3]:
# Gerar 24 combinações (4 x 2 x 3) via Python
combos = list(itertools.product(["None", "Low", "Medium", "High"], [False, True], ["Express", "Standard", "Economy"]))

schema = StructType([
    StructField("DiscountBand", StringType(),  False),
    StructField("IsHighValue",  BooleanType(), False),
    StructField("ShipmentMode", StringType(),  False),
])
from pyspark.sql import functions as F
flags_df = spark.createDataFrame(combos, schema)
flags_df = flags_df.withColumn("OrderFlagsSK", (F.monotonically_increasing_id() + 1).cast(IntegerType()))
flags_df.createOrReplaceTempView("src_order_flags")

spark.sql("""
    CREATE TABLE IF NOT EXISTS gold.DimOrderFlags
    (OrderFlagsSK INT, DiscountBand STRING, IsHighValue BOOLEAN, ShipmentMode STRING)
    USING DELTA
""")
spark.sql("DELETE FROM gold.DimOrderFlags")
spark.sql("INSERT INTO gold.DimOrderFlags SELECT OrderFlagsSK, DiscountBand, IsHighValue, ShipmentMode FROM src_order_flags")

n = spark.sql("SELECT COUNT(*) AS n FROM gold.DimOrderFlags").collect()[0]["n"]
print(f"DimOrderFlags: {n} combinações")
assert n == 24
print("24 combinações OK")

DimOrderFlags: 24 combinações
24 combinações OK


In [4]:
try:
    spark.sql("ALTER TABLE gold.factsales ADD COLUMNS (OrderFlagsSK INT)")
    print("Coluna OrderFlagsSK adicionada")
except Exception as e:
    if "already exists" in str(e).lower():
        print("Coluna OrderFlagsSK já existe")
    else:
        raise

Coluna OrderFlagsSK adicionada


In [5]:
spark.sql("""
    MERGE INTO gold.factsales tgt
    USING (
        SELECT fs.SalesSK, dof.OrderFlagsSK
        FROM gold.factsales fs
        JOIN bronze.orders o ON o.OrderID = fs.OrderID
        JOIN gold.dimorderflags dof
          ON dof.DiscountBand  = CASE WHEN fs.Discount = 0     THEN 'None'
                                      WHEN fs.Discount <= 0.05 THEN 'Low'
                                      WHEN fs.Discount <= 0.15 THEN 'Medium'
                                      ELSE 'High' END
         AND dof.IsHighValue   = (fs.NetRevenue > 1000)
         AND dof.ShipmentMode  = CASE o.ShipVia WHEN 1 THEN 'Express'
                                                WHEN 2 THEN 'Standard'
                                                ELSE 'Economy' END
    ) src ON tgt.SalesSK = src.SalesSK
    WHEN MATCHED THEN UPDATE SET tgt.OrderFlagsSK = src.OrderFlagsSK
""")

null_count = spark.sql("SELECT COUNT(*) AS n FROM gold.factsales WHERE OrderFlagsSK IS NULL").collect()[0]["n"]
print(f"FactSales sem OrderFlagsSK: {null_count} (esperado 0)")
assert null_count == 0
print("Junk dimension atualizada OK")

FactSales sem OrderFlagsSK: 0 (esperado 0)
Junk dimension atualizada OK


In [6]:
spark.sql("""
    SELECT dof.DiscountBand, dof.ShipmentMode,
           COUNT(*) AS Transacoes,
           ROUND(SUM(fs.GrossRevenue), 2) AS GrossRevenue,
           ROUND(SUM(fs.NetRevenue), 2)   AS NetRevenue
    FROM gold.factsales fs
    JOIN gold.dimorderflags dof ON dof.OrderFlagsSK = fs.OrderFlagsSK
    GROUP BY dof.DiscountBand, dof.ShipmentMode
    ORDER BY dof.DiscountBand, dof.ShipmentMode
""").show()

+------------+------------+----------+------------+----------+
|DiscountBand|ShipmentMode|Transacoes|GrossRevenue|NetRevenue|
+------------+------------+----------+------------+----------+
|        High|     Economy|      1290|   815501.64| 766810.94|
|        High|     Express|      1292|   747966.38| 697679.87|
|        High|    Standard|      1728|  1145449.16|1067095.27|
|         Low|     Economy|      1290|   815501.64| 766810.94|
|         Low|     Express|      1292|   747966.38| 697679.87|
|         Low|    Standard|      1728|  1145449.16|1067095.27|
|      Medium|     Economy|      1290|   815501.64| 766810.94|
|      Medium|     Express|      1292|   747966.38| 697679.87|
|      Medium|    Standard|      1728|  1145449.16|1067095.27|
|        None|     Economy|      1290|   815501.64| 766810.94|
|        None|     Express|      1292|   747966.38| 697679.87|
|        None|    Standard|      1728|  1145449.16|1067095.27|
+------------+------------+----------+------------+----